# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [4]:
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [5]:
# calorie_lookup_tool('bananas')
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [8]:
# Add news function tools 

# First import the data 

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")
nutrition_qna.count()

2289

In [9]:
@function_tool
def nutrition_tools(query: str, max_results :int)-> str :
    """ 
        Tool function for a RAG database which has an objective to answers to the question 
        of the peoples based on another one Question and Answer database 

        Args: 
         - query : str : query of the user 
         - max_results :int: the maximum number of response
        Returns : 
            Return the responses bases on our database
    """

    chroma_client = chromadb.PersistentClient("../chroma")
    nutrition_qna = chroma_client.get_collection(name="nutrition_qna")
    
    # Test query 1: Search for malnutrition symptoms
    print("=== Query: pregnancy ===")
    responses_list = []
    results = nutrition_qna.query(query_texts=[query], n_results=3)
    for i, doc in enumerate(results["documents"][0]):
        answer = doc.split('?\n')[1]
        responses_list.append(f"The Answers are {answer}") 

    return "Nutrition advice can be:\n" + "\n".join(responses_list)

In [10]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool and advice from nutrition_tools
    """,
    tools=[calorie_lookup_tool, nutrition_tools]
)

In [12]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "As a pregnant woman what is your advice about apple ?",
    )
    print(result.final_output)

Short answer: apples are safe and nutritious in pregnancy.

Tips:
- They’re high in fiber and vitamin C, which can help with digestion and immunity.
- Eat the skin (wash well) for extra fiber and nutrients.
- Wash or peel to reduce pesticide residues; opt for organic if you’re concerned, and avoid unpasteurized apple juice.
- Watch portions if you’re managing blood sugar; a medium apple is a reasonable daily option.
- If you have acid reflux, apples can irritate symptoms for some people—watch how you feel.
- If you have allergies or latex-fruit syndrome, check with your clinician.

If you want calories for a specific apple variety or preparation (raw vs baked), I can look that up.
